# Seguimiento de experimentos con MLflow

**Proyecto:** Predicción de abandono de clientes (Telco Customer Churn)  
**Etapa:** Experiment tracking local

## Objetivo

Registrar en MLflow los parámetros, métricas y artefactos de los modelos baseline del notebook `03_modelo_baseline_churn.ipynb`.

Se usó un Tracking Server local con SQLite. El seguimiento de experimentos no constituye un despliegue en producción.

## Alcance y relación con los notebooks anteriores

Los notebooks 01 y 02 contienen el EDA y la preparación; el notebook 03, el baseline. Aquí se reprodujo su partición para registrar resultados, sin repetir la exploración ni la limpieza.

Se registraron `DummyClassifier` (`most_frequent`) y regresión logística. Ajuste de hiperparámetros, Model Registry, API y despliegue en la nube quedan fuera de esta etapa.

## 1. Requisitos y servidor local

Se utilizó MLflow 3.17.0. El servidor local almacenó los metadatos en `mlflow.db` y los artefactos en `mlruns`.

Comando de inicio del servidor:

```powershell
& ".\.venv\Scripts\mlflow.exe" server --host 127.0.0.1 --port 5000 --backend-store-uri "sqlite:///mlflow.db" --default-artifact-root ".\mlruns" --allowed-hosts "localhost,127.0.0.1,127.0.0.1:5000"
```

La interfaz se verificó en `http://127.0.0.1:5000`.

## 2. Verificación de MLflow

La importación fue correcta y la versión instalada fue `3.17.0`.

In [16]:
import mlflow

print("Versión de MLflow:", mlflow.__version__)

Versión de MLflow: 3.17.0


## 3. Conexión con el Tracking Server

Se configuró y verificó el URI de seguimiento `http://127.0.0.1:5000`.

In [17]:

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI activo:", mlflow.get_tracking_uri())

Tracking URI activo: http://127.0.0.1:5000


## 4. Crear el experimento

Se creó el experimento `telco-churn-baseline` (ID `1`) para agrupar las ejecuciones.

In [18]:
experimento = mlflow.set_experiment("telco-churn-baseline")

print("Experimento:", experimento.name)
print("ID:", experimento.experiment_id)

Experimento: telco-churn-baseline
ID: 1


## 5. Información registrada por ejecución

En cada ejecución se registraron:

- **Parámetros:** nombre del modelo, estrategia de imputación, estrategia de codificación, tamaño de prueba, semilla y configuración relevante del estimador.
- **Métricas:** accuracy, precision, recall, F1 y ROC-AUC, con `Yes` como clase positiva.
- **Etiquetas:** `etapa=baseline` y `dataset=telco-churn`.
- **Artefactos:** matriz de confusión de cada modelo, guardada como imagen.

Se mantuvieron la partición y el preprocesamiento del notebook 03.

## 6. Registrar las ejecuciones del baseline

Se reprodujo la partición estratificada del notebook 03 (`test_size=0.2`, `random_state=31`) y se entrenaron ambos pipelines. Las métricas coincidieron con el baseline.

Se registró cada modelo en un run independiente con parámetros, cinco métricas (`accuracy`, `precision`, `recall`, `f1`, `roc_auc`) y etiquetas de etapa y dataset.

Las matrices de confusión se guardaron como PNG en `matrices_confusion` dentro de cada run.

In [19]:

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ruta_preparada = (
    Path.cwd()
    / "data"
    / "processed"
    / "telco_churn_preparado.csv"
)

if not ruta_preparada.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset preparado: {ruta_preparada}"
    )

df = pd.read_csv(ruta_preparada)
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=31,
    stratify=y,
)

print("Dataset:", df.shape)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Distribución de Churn en prueba (%):")
print(y_test.value_counts(normalize=True).mul(100).round(3))


Dataset: (7043, 21)
X_train: (5634, 19) | X_test: (1409, 19)
Distribución de Churn en prueba (%):
Churn
No     73.456
Yes    26.544
Name: proportion, dtype: float64


In [20]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

columnas_numericas = X_train.select_dtypes(include="number").columns.tolist()
columnas_categoricas = X_train.select_dtypes(
    exclude="number"
).columns.tolist()

pipeline_numerico = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

pipeline_categorico = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore")),
])

preprocesador = ColumnTransformer(transformers=[
    ("numericas", pipeline_numerico, columnas_numericas),
    ("categoricas", pipeline_categorico, columnas_categoricas),
])

modelo_dummy = Pipeline(steps=[
    ("preprocesador", clone(preprocesador)),
    ("modelo", DummyClassifier(strategy="most_frequent")),
])

modelo_logistico = Pipeline(steps=[
    ("preprocesador", clone(preprocesador)),
    ("modelo", LogisticRegression(max_iter=1000, random_state=31)),
])

modelo_dummy.fit(X_train, y_train)
modelo_logistico.fit(X_train, y_train)

pred_dummy = modelo_dummy.predict(X_test)
pred_logistico = modelo_logistico.predict(X_test)

def calcular_metricas(modelo, predicciones):
    probabilidades = modelo.predict_proba(X_test)
    indice_yes = list(modelo.classes_).index("Yes")

    return {
        "accuracy": accuracy_score(y_test, predicciones),
        "precision": precision_score(
            y_test, predicciones, pos_label="Yes", zero_division=0
        ),
        "recall": recall_score(
            y_test, predicciones, pos_label="Yes", zero_division=0
        ),
        "f1": f1_score(
            y_test, predicciones, pos_label="Yes", zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test.eq("Yes").astype(int),
            probabilidades[:, indice_yes],
        ),
    }

metricas = pd.DataFrame(
    {
        "Dummy": calcular_metricas(modelo_dummy, pred_dummy),
        "Regresión logística": calcular_metricas(
            modelo_logistico, pred_logistico
        ),
    }
).T

metricas.round(3)

,accuracy,precision,recall,f1,roc_auc
Dummy,0.735,0.000,0.0,0.000,0.500
Regresión logística,0.784,0.613,0.5,0.551,0.826


In [21]:
with mlflow.start_run(run_name="dummy_baseline"):
    mlflow.log_params({
        "modelo": "DummyClassifier",
        "estrategia": "most_frequent",
        "test_size": 0.2,
        "random_state": 31,
        "imputacion_numerica": "median",
        "imputacion_categorica": "most_frequent",
        "codificacion": "OneHotEncoder",
    })

    mlflow.log_metrics(
        metricas.loc["Dummy"].to_dict()
    )

    mlflow.set_tags({
        "etapa": "baseline",
        "dataset": "telco-churn",
    })

print("Ejecución Dummy registrada.")

🏃 View run dummy_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/25a5c0bdd99145a68bb4cb90a6e6da1c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Ejecución Dummy registrada.


In [22]:
with mlflow.start_run(run_name="logistic_regression_baseline"):
    mlflow.log_params({
        "modelo": "LogisticRegression",
        "max_iter": 1000,
        "random_state_modelo": 31,
        "test_size": 0.2,
        "random_state_split": 31,
        "imputacion_numerica": "median",
        "imputacion_categorica": "most_frequent",
        "codificacion": "OneHotEncoder",
    })

    mlflow.log_metrics(
        metricas.loc["Regresión logística"].to_dict()
    )

    mlflow.set_tags({
        "etapa": "baseline",
        "dataset": "telco-churn",
    })

print("Ejecución de regresión logística registrada.")


🏃 View run logistic_regression_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/14d22bfc87314195b33030ff5d103cf5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Ejecución de regresión logística registrada.


In [23]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import ConfusionMatrixDisplay

carpeta_artifacts = Path("artifacts")
carpeta_artifacts.mkdir(exist_ok=True)

client = MlflowClient()
runs = client.search_runs(
    experiment_ids=[experimento.experiment_id]
)

runs_por_nombre = {
    run.data.tags.get("mlflow.runName"): run.info.run_id
    for run in runs
}

predicciones = {
    "dummy_baseline": ("Dummy", pred_dummy),
    "logistic_regression_baseline": (
        "Regresión logística",
        pred_logistico,
    ),
}

for nombre_run, (titulo, predicciones_modelo) in predicciones.items():
    figura, eje = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predicciones_modelo,
        labels=["No", "Yes"],
        display_labels=["No (permaneció)", "Yes (abandonó)"],
        cmap="Blues",
        values_format="d",
        ax=eje,
        colorbar=False,
    )

    eje.set_title(f"Matriz de confusión: {titulo}")
    figura.tight_layout()

    ruta_imagen = carpeta_artifacts / f"matriz_confusion_{nombre_run}.png"
    figura.savefig(ruta_imagen, dpi=150)
    plt.close(figura)

    if nombre_run not in runs_por_nombre:
        raise ValueError(f"No se encontró el run de MLflow: {nombre_run}")

    client.log_artifact(
        runs_por_nombre[nombre_run],
        str(ruta_imagen.resolve()),
        artifact_path="matrices_confusion",
    )

    print(f"Artefacto registrado para {nombre_run}: {ruta_imagen}")

Artefacto registrado para dummy_baseline: artifacts\matriz_confusion_dummy_baseline.png
Artefacto registrado para logistic_regression_baseline: artifacts\matriz_confusion_logistic_regression_baseline.png


## 7. Verificar los resultados en la interfaz

La interfaz mostró los runs `dummy_baseline` y `logistic_regression_baseline`. Sus métricas coincidieron con el notebook 03 y ambas matrices aparecieron en Artifacts.

## 8. Registro y prueba del modelo

El pipeline de regresión logística se guardó en el run existente de MLflow. Se cargó mediante su URI y se probaron cinco registros: las cinco predicciones coincidieron con las etiquetas reales. La tabla de predicciones se guardó como CSV y se registró como artefacto del run.

Al inferir el esquema del modelo, MLflow mostró una advertencia sobre columnas enteras que podrían contener faltantes. El guardado y la prueba funcionaron; esta advertencia deberá revisarse antes de ofrecer predicciones sobre datos externos.

In [25]:
run_id_logistico = runs_por_nombre["logistic_regression_baseline"]

with mlflow.start_run(run_id=run_id_logistico):
    info_modelo = mlflow.sklearn.log_model(
        sk_model=modelo_logistico,
        name="modelo",
        input_example=X_test.head(3),
        skops_trusted_types=["numpy.dtype"],
    )

print("Modelo guardado en:", info_modelo.model_uri)

c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run logistic_regression_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/36267eb0733d46be8564a4fe9607295b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Modelo guardado en: models:/m-f160971b2ffc4589b97a6536c495746b


In [26]:
modelo_cargado = mlflow.pyfunc.load_model(info_modelo.model_uri)

muestra = X_test.head(5)
predicciones_cargadas = modelo_cargado.predict(muestra)

comparacion = pd.DataFrame({
    "Churn real": y_test.head(5).values,
    "Churn predicho": predicciones_cargadas,
})

comparacion

,Churn real,Churn predicho
0,No,No
1,Yes,Yes
2,No,No
3,No,No
4,No,No


In [27]:
from pathlib import Path
from mlflow.tracking import MlflowClient

ruta_csv = Path("artifacts") / "predicciones_muestra.csv"
ruta_csv.parent.mkdir(exist_ok=True)

comparacion.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

run_id_logistico = runs_por_nombre["logistic_regression_baseline"]
client = MlflowClient()

client.log_artifact(
    run_id_logistico,
    str(ruta_csv.resolve()),
    artifact_path="predicciones",
)

print("Tabla registrada en MLflow:", ruta_csv)

Tabla registrada en MLflow: artifacts\predicciones_muestra.csv


# 9. Conclusiones

El experimento `telco-churn-baseline` contiene dos runs con parámetros, métricas, etiquetas y matrices de confusión. Las métricas coinciden con el notebook 03: la regresión logística obtuvo `accuracy=0.784`, `F1=0.551` y `ROC-AUC=0.826`; el Dummy alcanzó `accuracy=0.735` y `ROC-AUC=0.500`.

El pipeline logístico y la tabla de cinco predicciones también se registraron en MLflow. Las cinco predicciones de prueba coincidieron con sus etiquetas reales; esta comprobación funcional no sustituye la evaluación completa ni constituye un despliegue de producción.

La siguiente etapa es automatizar el entrenamiento y la evaluación con Prefect.